# 参数管理

在选择了架构并设置了超参数后，我们就进入了训练阶段。
此时，我们的目标是找到使损失函数最小化的模型参数值。
经过训练后，我们将需要使用这些参数来做出未来的预测。
此外，有时我们希望提取参数，以便在其他环境中复用它们，
将模型保存下来，以便它可以在其他软件中执行，
或者为了获得科学的理解而进行检查。

之前的介绍中，我们只依靠深度学习框架来完成训练的工作，
而忽略了操作参数的具体细节。
本节，我们将介绍以下内容：

* 访问参数，用于调试、诊断和可视化；
* 参数初始化；
* 在不同模型组件间共享参数。

(**我们首先看一下具有单隐藏层的多层感知机。**)


In [2]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[ 0.0384],
        [-0.0640]], grad_fn=<AddmmBackward0>)

## [**参数访问**]

我们从已有模型中访问参数。
当通过`Sequential`类定义模型时，
我们可以通过索引来访问模型的任意层。
这就像模型是一个列表一样，每层的参数都在其属性中。
如下所示，我们可以检查第二个全连接层的参数。


In [3]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.2427,  0.1369, -0.3005,  0.1671, -0.1098, -0.1399,  0.0461,  0.2825]])), ('bias', tensor([-0.0293]))])


输出的结果告诉我们一些重要的事情：
首先，这个全连接层包含两个参数，分别是该层的权重和偏置。
两者都存储为单精度浮点数（float32）。
注意，参数名称允许唯一标识每个参数，即使在包含数百个层的网络中也是如此。

### [**目标参数**]

注意，每个参数都表示为参数类的一个实例。
要对参数执行任何操作，首先我们需要访问底层的数值。
有几种方法可以做到这一点。有些比较简单，而另一些则比较通用。
下面的代码从第二个全连接层（即第三个神经网络层）提取偏置，
提取后返回的是一个参数类实例，并进一步访问该参数的值。


In [4]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.0293], requires_grad=True)
tensor([-0.0293])


In [5]:
#Parameter containing:
#tensor([0.0524], requires_grad=True)
#tensor([0.0524])
# Parameter containing说明它是模型参数。
#requires_grad=True
# 表示 PyTorch 会追踪与该参数相关的计算。调用反向传播后，会计算损失函数对它的梯度：
#net[2].bias.grad

In [6]:
#现代 PyTorch 中，读取参数值更推荐使用：net[2].bias.detach()
#而不是 .data
#因为直接通过 .data 修改参数可能绕过自动求导机制，造成难以发现的梯度问题。
#如果确实需要手动修改参数，推荐：
#with torch.no_grad():
#    net[2].bias.fill_(0)

参数是复合的对象，包含值、梯度和额外信息。
这就是我们需要显式参数值的原因。
除了值之外，我们还可以访问每个参数的梯度。
**在上面这个网络中，由于我们还没有调用反向传播，所以参数的梯度处于初始状态**。


In [7]:
net[2].weight.grad == None

True

### [**一次性访问所有参数**]

当我们需要对所有参数执行操作时，逐个访问它们可能会很麻烦。
当我们处理更复杂的块（例如，嵌套块）时，情况可能会变得特别复杂，
因为我们需要递归整个树来提取每个子块的参数。
下面，我们将通过演示来比较访问第一个全连接层的参数和访问所有层。


In [8]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [9]:
#为什么print(*[(name, param.shape) for name, param in net[0].named_parameters()])和print(*[(name, param.shape) for name, param in net.named_parameters()])的输出结果不同呢？
#有一个*号
# 列表推导式首先产生：
#items = [
#    ("weight", torch.Size([8, 4])),
#    ("bias", torch.Size([8]))
#]
#如果不加 *：print(items)
#会把整个列表作为一个对象打印：
# [('weight', torch.Size([8, 4])), ('bias', torch.Size([8]))]
#加上 *：
#会把列表拆开，打印每个元素：       ##相当于print(items[0], items[1])
#('weight', torch.Size([8, 4]))
#('bias', torch.Size([8]))
#也可以指定每个元素之间用换行分隔：
# print(*items, sep='\n')

#总结
#print(items)   # 打印整个容器
#print(*items)  # 取出容器中的所有元素，再分别传给print


这为我们提供了另一种访问网络参数的方式，如下所示。


In [10]:
net.state_dict()['2.bias'].data

tensor([-0.0293])

### [**从嵌套块收集参数**]

让我们看看，如果我们将多个块相互嵌套，参数命名约定是如何工作的。
我们首先定义一个生成块的函数（可以说是“块工厂”），然后将这些块组合到更大的块中。


In [11]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
        #这里每个 block1() 都是独立创建的，所以四个块的参数不共享
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.4537],
        [-0.4537]], grad_fn=<AddmmBackward0>)

[**设计了网络后，我们看看它是如何工作的。**]


In [12]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


因为层是分层嵌套的，所以我们也可以像通过嵌套列表索引一样访问它们。
下面，我们访问第一个主要的块中、第二个子块的第一层的偏置项。


In [13]:
print(rgnet[0][1][0].bias.shape)   ##jupyter默认只显示最后一行，想都显示要用print
rgnet[0][1][0].bias.data

torch.Size([8])


tensor([ 0.0410, -0.2488,  0.4825,  0.4449, -0.3397, -0.1266, -0.0635, -0.3296])

In [14]:
display(rgnet[0][1][0].bias.shape)
display(rgnet[0][1][0].bias.data)

torch.Size([8])

tensor([ 0.0410, -0.2488,  0.4825,  0.4449, -0.3397, -0.1266, -0.0635, -0.3296])

In [15]:
#或者把两者组成一个元组，作为最后一个表达式：
rgnet[0][1][0].bias.shape, rgnet[0][1][0].bias.data

(torch.Size([8]),
 tensor([ 0.0410, -0.2488,  0.4825,  0.4449, -0.3397, -0.1266, -0.0635, -0.3296]))

## 参数初始化

知道了如何访问参数后，现在我们看看如何正确地初始化参数。
我们在 :numref:`sec_numerical_stability`中讨论了良好初始化的必要性。
深度学习框架提供默认随机初始化，
也允许我们创建自定义初始化方法，
满足我们通过其他规则实现初始化权重。


默认情况下，PyTorch会根据一个范围均匀地初始化权重和偏置矩阵，
这个范围是根据输入和输出维度计算出的。
PyTorch的`nn.init`模块提供了多种预置初始化方法。


### [**内置初始化**]

让我们首先调用内置的初始化器。
下面的代码将所有权重参数初始化为标准差为0.01的高斯随机变量，
且将偏置参数设置为0。


In [16]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0059, -0.0016, -0.0019,  0.0025]), tensor(0.))

我们还可以将所有参数初始化为给定的常数，比如初始化为1。


In [17]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

我们还可以[**对某些块应用不同的初始化方法**]。
例如，下面我们使用Xavier初始化方法初始化第一个神经网络层，
然后将第三个神经网络层初始化为常量值42。


In [18]:
print(net)
print(net[0].weight.data[0])
print(net[2].weight.data)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)
tensor([1., 1., 1., 1.])
tensor([[1., 1., 1., 1., 1., 1., 1., 1.]])


In [19]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)            ##将net的第一个线性层的权重参数初始化为Xavier初始化
net[2].apply(init_42)                ##将net的第三个线性层的权重参数初始化为常数42
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([0.6588, 0.4707, 0.0126, 0.4131])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### [**自定义初始化**]

有时，深度学习框架没有提供我们需要的初始化方法。
在下面的例子中，我们使用以下的分布为任意权重参数$w$定义初始化方法：

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \text{ 可能性 } \frac{1}{4} \\
            0    & \text{ 可能性 } \frac{1}{2} \\
        U(-10, -5) & \text{ 可能性 } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


同样，我们实现了一个`my_init`函数来应用到`net`。


In [20]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5        ##>=此处是一个布尔索引，表示只保留大于等于5的权重值，小于5的权重值将被置为0

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[9.3425, 0.0000, -0.0000, -0.0000],
        [9.1709, 0.0000, -0.0000, 8.8751]], grad_fn=<SliceBackward0>)

注意，我们始终可以直接设置参数。


In [21]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.,  1.,  1.,  1.])

## [**参数绑定**]

有时我们希望在多个层间共享参数：
我们可以定义一个稠密层，然后使用它的参数来设置另一个层的参数。


In [27]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值，并且会同步更新
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


这个例子表明第三个和第五个神经网络层的参数是绑定的。
它们不仅值相等，而且由相同的张量表示。
因此，如果我们改变其中一个参数，另一个参数也会改变。
这里有一个问题：当参数绑定时，梯度会发生什么情况？
答案是由于模型参数包含梯度，因此在反向传播期间第二个隐藏层
（即第三个神经网络层）和第三个隐藏层（即第五个神经网络层）的梯度会加在一起。
【参数绑定常常用来减少参数量、防止过拟合、权重共享】



In [28]:
# 演示：绑定时，两个位置的梯度会“相加”到同一份参数上
# 思路：构造两个完全等价（每一层权重都相同）的网络——
# 网络1：参数绑定（两个位置共用 shared）
# 网络2：不绑定（两个独立层）
# 在相同输入下反向传播后，shared 的梯度应该等于两个独立层梯度之和

torch.manual_seed(0)

# 网络1：绑定
shared = nn.Linear(8, 8)
net1 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                     shared, nn.ReLU(),
                     shared, nn.ReLU(),
                     nn.Linear(8, 1))

# 网络2：不绑定，并把“每一层”的权重复制为与 net1 完全一致，保证公平对比
net2 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                     nn.Linear(8, 8), nn.ReLU(),
                     nn.Linear(8, 8), nn.ReLU(),
                     nn.Linear(8, 1))
net2[0].weight.data.copy_(net1[0].weight.data)
net2[0].bias.data.copy_(net1[0].bias.data)
net2[2].weight.data.copy_(shared.weight.data)
net2[2].bias.data.copy_(shared.bias.data)
net2[4].weight.data.copy_(shared.weight.data)
net2[4].bias.data.copy_(shared.bias.data)
net2[6].weight.data.copy_(net1[6].weight.data)
net2[6].bias.data.copy_(net1[6].bias.data)

# 相同输入（4 个样本，每个样本 4 个特征，匹配第一层 Linear(4, 8)）
X = torch.randn(4, 4)

net1(X).sum().backward()
net2(X).sum().backward()

# 提取各梯度
grad_shared = shared.weight.grad          # 绑定网络中 shared 的梯度
grad_a = net2[2].weight.grad              # 不绑定网络中“第3层”的梯度
grad_b = net2[4].weight.grad              # 不绑定网络中“第5层”的梯度

print("shared 的梯度 == 独立层A 的梯度？", torch.allclose(grad_shared, grad_a))
print("shared 的梯度 == 独立层B 的梯度？", torch.allclose(grad_shared, grad_b))
print("shared 的梯度 == A + B 之和？   ", torch.allclose(grad_shared, grad_a + grad_b))

shared 的梯度 == 独立层A 的梯度？ False
shared 的梯度 == 独立层B 的梯度？ False
shared 的梯度 == A + B 之和？    True


## 补充：常用的深度学习参数初始化方法

参数初始化的目标是让前向传播的激活值和反向传播的梯度保持在合适范围，避免过早出现梯度消失或梯度爆炸。

| 初始化方法 | 适用场景 | PyTorch |
|---|---|---|
| 零初始化 | 偏置、某些特殊输出层 | `nn.init.zeros_` |
| 常数初始化 | 偏置、BatchNorm参数或特殊实验 | `nn.init.constant_` |
| 普通随机初始化 | 简单实验、小范围权重 | `normal_`、`uniform_` |
| Xavier初始化 | `tanh`、`sigmoid`或近似线性激活 | `xavier_uniform_`、`xavier_normal_` |
| Kaiming/He初始化 | `ReLU`、`LeakyReLU` | `kaiming_uniform_`、`kaiming_normal_` |
| 正交初始化 | RNN、循环矩阵、深层线性变换 | `orthogonal_` |
| 截断正态初始化 | Transformer、ViT等架构中较常见 | `trunc_normal_` |


### 1. 零初始化

零初始化通常用于偏置。一般不要把隐藏层的所有权重都初始化为零，否则同一层的神经元会得到相同的输出和梯度，无法打破对称性。偏置初始化为零通常没有这个问题，因为权重已经是随机的。


In [ ]:
zero_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_zero_bias(m):
    if isinstance(m, nn.Linear) and m.bias is not None:
        nn.init.zeros_(m.bias)

zero_net.apply(init_zero_bias)
zero_net[0].weight.data[0], zero_net[0].bias.data

### 2. 常数初始化

常数初始化将参数设置为指定值，主要用于特殊结构或实验。隐藏层权重通常不应全部初始化成相同常数，否则仍然存在对称性问题。


In [ ]:
constant_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_constant(m):
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

constant_net.apply(init_constant)
constant_net[0].weight.data[0], constant_net[0].bias.data

### 3. 正态分布初始化

权重从指定的正态分布中采样。标准差太小可能使信号和梯度越来越小；标准差太大可能导致激活值和梯度爆炸。下面使用均值0、标准差0.01的正态分布。


In [ ]:
normal_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

normal_net.apply(init_normal)
normal_net[0].weight.data.mean(), normal_net[0].weight.data.std()

### 4. 均匀分布初始化

权重从指定的均匀分布区间中采样。和普通正态初始化一样，区间大小需要合理选择。下面使用区间$[-0.1, 0.1]$。


In [ ]:
uniform_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_uniform(m):
    if isinstance(m, nn.Linear):
        nn.init.uniform_(m.weight, -0.1, 0.1)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

uniform_net.apply(init_uniform)
uniform_net[0].weight.data.min(), uniform_net[0].weight.data.max()

### 5. Xavier初始化

Xavier初始化也叫Glorot初始化，它同时考虑输入特征数`fan_in`和输出特征数`fan_out`，通常适用于`tanh`、`sigmoid`和近似线性的激活函数。Xavier正态初始化的标准差大致为：

$$
\sqrt{\frac{2}{\text{fan\_in}+\text{fan\_out}}}
$$


In [ ]:
xavier_net = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 1))

def init_xavier(m):
    if isinstance(m, nn.Linear):
        gain = nn.init.calculate_gain('tanh')
        nn.init.xavier_uniform_(m.weight, gain=gain)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

xavier_net.apply(init_xavier)
xavier_net[0].weight.data[0]

### 6. Kaiming初始化

Kaiming初始化也叫He初始化，主要为`ReLU`和`LeakyReLU`等激活函数设计。对于ReLU，Kaiming正态初始化的标准差大致为：

$$
\sqrt{\frac{2}{\text{fan\_in}}}
$$


In [ ]:
kaiming_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_kaiming(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(
            m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

kaiming_net.apply(init_kaiming)
kaiming_net[0].weight.data[0]

### 7. 正交初始化

正交初始化让权重矩阵具有正交性质，有助于在多次矩阵变换中保持信号尺度，常见于RNN、LSTM、GRU的循环权重以及某些深层线性网络。


In [ ]:
orthogonal_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_orthogonal(m):
    if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

orthogonal_net.apply(init_orthogonal)
W = orthogonal_net[0].weight.data
W.T @ W

### 8. 截断正态初始化

截断正态分布会限制随机数不能过度偏离均值，常见于一些Transformer和视觉Transformer架构。具体标准差应优先遵循对应模型的官方实现。


In [ ]:
truncated_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

def init_truncated_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(
            m.weight, mean=0, std=0.02, a=-0.04, b=0.04)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

truncated_net.apply(init_truncated_normal)
truncated_net[0].weight.data.min(), truncated_net[0].weight.data.max()

### 常见选择原则

- `ReLU`、`LeakyReLU`：通常使用Kaiming初始化。
- `tanh`、`sigmoid`：通常使用Xavier初始化。
- RNN循环权重：常考虑正交初始化。
- 偏置：通常初始化为0。
- 已有预训练模型：不要重新初始化。
- 特定成熟架构：优先遵循其官方初始化方法。

对于本节的`Linear + ReLU`网络，典型选择是权重使用Kaiming初始化，偏置初始化为零。


## 小结

* 我们有几种方法可以访问、初始化和绑定模型参数。
* 我们可以使用自定义初始化方法。

## 练习

1. 使用 :numref:`sec_model_construction` 中定义的`FancyMLP`模型，访问各个层的参数。
1. 查看初始化模块文档以了解不同的初始化方法。
1. 构建包含共享参数层的多层感知机并对其进行训练。在训练过程中，观察模型各层的参数和梯度。
1. 为什么共享参数是个好主意？


[Discussions](https://discuss.d2l.ai/t/1829)
